In [1]:
# 加载环境变量
from dotenv import load_dotenv

load_dotenv()

True

# 1.消息类型

在LangChain中，发送给LLM的消息、LLM返回的消息都统一被封装为BaseMessage，它是Agent中基本的上下文单元。

在LangChain中，我们并不需要自己创建BaseMessage对象，LangChain已经把常见消息根据角色（Role）创建了对应的BaseMessage的子类：
- SystemMessage：role是system，代表系统消息，用于设定模型角色和交互背景
- HumanMessage：role是user，代表用户输入的消息
- AIMessage：role是assistant，代表LLM生成的响应，包含：文本、工具调用、元数据
- ToolMessage：role是tool，代表工具调用时产生的结果

我们可以直接使用这些Messages类型来发送消息。

In [2]:
from langchain.agents import create_agent
from langchain.tools import tool
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

# 定义工具
@tool
def get_weather(location: str) -> str:
    """
    Get the weather in a given location.
    Args:
        location: city name or coordinates
    """
    return f"Current weather in {location} is sunny"

# 创建Agent
agent = create_agent(model="deepseek-chat", tools=[get_weather])

# 调用Agent，发送消息
response = agent.invoke({
    "messages": [
        SystemMessage("请使用工具来获取天气信息。"),
        HumanMessage("你好，我是虎哥."),
        AIMessage("你好，虎哥，很高兴认识你."),
        HumanMessage("北京今天天气如何？")
    ]
})

print(response)

D:\CODE\jc-course\.venv\Lib\site-packages\langgraph\checkpoint\serde\encrypted.py:5: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


{'messages': [SystemMessage(content='请使用工具来获取天气信息。', additional_kwargs={}, response_metadata={}, id='c8f72bd2-df91-4b0e-9020-957472188859'), HumanMessage(content='你好，我是虎哥.', additional_kwargs={}, response_metadata={}, id='508dd95b-8f02-4459-8312-16b6800b5aaa'), AIMessage(content='你好，虎哥，很高兴认识你.', additional_kwargs={}, response_metadata={}, id='65ed0bf2-9081-4bc7-883b-2658fb2137e5', tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='北京今天天气如何？', additional_kwargs={}, response_metadata={}, id='855718ec-62b2-47d9-a000-53ad23c2e0c8'), AIMessage(content='我来帮你查一下北京今天的天气！', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 52, 'prompt_tokens': 316, 'total_tokens': 368, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 316}, 'model_provider': 'deepseek', 'model_name': 'deepseek-v4-flash', 'system_fingerprint': 'fp_8b330d02d0_prod0820

In [3]:
for message in response['messages']:
    message.pretty_print()

================================ System Message ================================

请使用工具来获取天气信息。
================================ Human Message =================================

你好，我是虎哥.
================================== Ai Message ==================================

你好，虎哥，很高兴认识你.
================================ Human Message =================================

北京今天天气如何？
================================== Ai Message ==================================

我来帮你查一下北京今天的天气！
Tool Calls:
  get_weather (call_00_u2kX2N9ZcnLi2rYWk6BN1224)
 Call ID: call_00_u2kX2N9ZcnLi2rYWk6BN1224
  Args:
    location: 北京
================================= Tool Message =================================
Name: get_weather

Current weather in 北京 is sunny
================================== Ai Message ==================================

虎哥，北京今天天气是**晴天（sunny）** ☀️，阳光不错，适合出去走走！请问还需要了解其他城市的天气吗？😊


# 2.多模态消息

之前我们都是向模型发送文本消息，但是 LangChain 也支持向模型发送多模态消息，比如图片、音频、视频、文本等。但前提是必须是多模态模型才支持。

一些支持多模态的模型有：
- qwen3.5-plus
- gpt-5-nano
- ...

我们以qwen3.5-plus为例，演示向模型发送图片消息

## 2.1.在线图片
首先，我们演示如何发送一个在线图片给模型，也就是指定模型的url地址。
图片如下：

<img src="https://help-static-aliyun-doc.aliyuncs.com/file-manage-files/zh-CN/20241022/emyrja/dog_and_girl.jpeg" width="500" height="300" alt="图片描述">



In [35]:
from langchain.chat_models import init_chat_model
import os

# 初始化模型
model = init_chat_model(
    model="qwen3.5-omni-plus",  # 模型名称，这里选择qwen3.5-plus，这是一个多模态模型，支持图片、文本、音频、视频
    model_provider="openai",
    base_url=os.getenv("DASHSCOPE_BASE_URL"),
    api_key=os.getenv("DASHSCOPE_API_KEY")
)

In [36]:
# 创建Agent
agent = create_agent(model=model)

In [39]:
# 准备多模态消息
message = HumanMessage([
        {"type": "text", "text": "描述一下这个图."},
        {"type": "image", "url": "https://help-static-aliyun-doc.aliyuncs.com/file-manage-files/zh-CN/20241022/emyrja/dog_and_girl.jpeg"},
    ])

In [40]:
stream = agent.stream(
    {"messages": [message]},
    stream_mode="messages"
)
for chunk, metadata in stream:
    if chunk.content:
        print(chunk.content, end="", flush=True)

这张图片展现了一个温馨、宁静的海滩场景，充满人与宠物之间的亲密互动。

**主体内容：**
- 一位年轻女性坐在沙滩上，面带微笑，侧身对着镜头，目光温柔地注视着她面前的狗。
- 她穿着一件黑白格子衬衫和深色裤子，赤脚踩在沙地上，左手腕戴着一块白色手表。
- 一只浅黄色的拉布拉多犬（或类似品种）坐在她对面，前爪抬起，正与女子的手掌轻轻相触，仿佛在“击掌”或握手。
- 狗狗佩戴着一条彩色图案的胸背带，旁边还放着一条红色的牵引绳。

**环境背景：**
- 场景位于海边沙滩，沙子细腻，上面有脚印和自然纹理。
- 远处是平静的海面，波浪轻柔拍打岸边，天空明亮，呈现出日出或日落时分的柔和暖光。
- 阳光从右侧斜射过来，在人物和狗狗身上形成温暖的轮廓光，营造出梦幻而浪漫的氛围。

**整体氛围：**
画面传递出一种宁静、幸福、陪伴的情感。人与宠物之间默契的互动，加上黄昏/黎明时分的柔光，使整张照片充满治愈感和生活气息，象征着忠诚、友谊与简单快乐的生活瞬间。

——  
这是一幅极具情感共鸣的摄影作品，适合用于表达宠物陪伴、户外休闲、人与自然和谐共处等主题。

## 2.2.本地图片数据
有时候用户会上传图片数据，而不是图片的url地址。我们需要将图片数据转换成base64字符串，然后发送给模型。

接下来我们会模拟图片上传、转换的过程。

首先，我们安装一个上传组件，用于模拟图片上传。


In [ ]:
!uv add ipywidgets

然后，我们创建一个上传组件，用于模拟图片上传。


In [41]:
from ipywidgets import FileUpload
from IPython.display import display

uploader = FileUpload(accept='*', multiple=False)
display(uploader)

FileUpload(value=(), accept='*', description='Upload')

In [42]:
print(uploader.value)

({'name': '屏幕截图 2025-07-08 214415.png', 'type': 'image/png', 'size': 8692524, 'content': <memory at 0x000001998F242A40>, 'last_modified': datetime.datetime(2025, 7, 8, 13, 44, 17, 35000, tzinfo=datetime.timezone.utc)},)


In [43]:
# 读取图片，转为base64字符串
import base64

# 获取第一个（也是唯一一个）上传的文件
uploaded_file = uploader.value[0]

# 获取其内存视图
content_mv = uploaded_file["content"]

# 转换内存视图->字节
img_bytes = bytes(content_mv)  # or content_mv.tobytes()

# base64编码
img_b64 = base64.b64encode(img_bytes).decode("utf-8")

In [44]:
# 组织多模态消息
multimodal_question = HumanMessage(content=[
    {
        "type": "image",
        "base64": img_b64,
        "mime_type": "image/jpeg",
    },
    {"type": "text", "text": "给我讲讲图片中的内容"}
])

for chunk, metadata in agent.stream(
    {"messages": [multimodal_question]},
    stream_mode="messages"
):
    print(chunk.content, end="", flush=True)

这张图片展示的是一个 **Windows 操作系统的电脑桌面截图**，背景是一张非常唯美的竹林小径风景图，整体风格清新自然、充满生机。

---

### 🖼️ 桌面壁纸内容：

- **主体场景**：一条笔直的小路穿过茂密的竹林，两侧是高耸入云的翠绿色竹子，形成天然的“绿色隧道”。
- **光线效果**：阳光从竹叶缝隙中洒下，营造出明亮通透、梦幻般的氛围。
- **前景细节**：小路两旁有金黄色的干草或芦苇，与翠绿竹林形成鲜明色彩对比。
- **远景人物**：在小路尽头隐约可见几个人影，增添了画面的空间感和生活气息。
- **拍摄角度**：采用低角度仰拍，强化了竹林的高大挺拔和纵深感。

> 这种壁纸常用于营造宁静、放松的工作或学习环境。

---

### 💻 桌面图标（左侧排列）：

从上到下依次是：

1. **回收站** —— Windows 系统默认图标。
2. **网易云音乐** —— 红色圆形带音符标志的音乐播放器。
3. **Microsoft Edge** —— 微软浏览器，蓝绿渐变波浪形图标。
4. **QQ** —— 腾讯即时通讯软件，企鹅头像。
5. **微信** —— 绿色对话气泡图标，中国最流行的社交应用。
6. **我的文件在哪里** —— 自定义文件夹或快捷方式，可能是用户自己命名的文档管理入口。
7. **NoteExpress** —— 文献管理软件，常用于学术研究者。
8. **PyCharm 2025.1.3** —— JetBrains 出品的 Python 集成开发环境（IDE），版本较新（注意：截至当前实际最新为 2024.x，此版本号可能为模拟或未来版）。

---

### 📱 任务栏（底部）：

- **开始菜单按钮**（Windows 徽标）
- **搜索框**：“搜索”字样 + 放大镜图标
- **固定/打开的应用程序图标**：
  - 文件资源管理器
  - Microsoft Edge 浏览器
  - 设置（齿轮图标）
  - 网易云音乐
  - 微信
  - Word（W 图标）
- **系统托盘区（右下角）**：
  - 输入法状态：“英”表示英文输入
  - 网络、音量、电池等系统图标
  - 时间日期：**2025年7月8日 21:44**

> 注意：这个时间点（2025年）表明这可能是一个未来